# check_zpools Quickstart

Monitor ZFS pool health from the CLI or from Python.

Install the package, then try the CLI and the offline API examples below. The pool-reading commands (`check`, `daemon`) need a real ZFS pool, so this notebook sticks to what runs anywhere.

## 1) Install latest from GitHub (default)

In [ ]:
import os
import sys
import sysconfig

# Install uv into this kernel's environment first: a bare `!uv` needs uv on
# PATH, which a reader running this notebook from a plain Python will not have.
# --python then pins the install to the kernel's own interpreter; without it uv
# picks the first environment it discovers and the import below fails.
!{sys.executable} -m pip install --quiet --upgrade uv
assert _exit_code == 0

!{sys.executable} -m uv pip install --python {sys.executable} --upgrade git+https://github.com/bitranox/check_zpools.git
assert _exit_code == 0

# Put this kernel's script directory first on PATH, so the `!` cells below run
# the console script that was just installed rather than whichever same-named
# command happens to sit on the ambient PATH.
os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ["PATH"]

## Or: Install from PyPI (alternative)
Uncomment and run the cell below if you prefer the PyPI release.

In [ ]:
# !{sys.executable} -m uv pip install --python {sys.executable} --upgrade check_zpools


## 2) Confirm the install

In [ ]:
from importlib.metadata import version

import check_zpools

# importlib.metadata reads the installed distribution, so this works whether
# or not the package re-exports a __version__ attribute.
print(f"check_zpools {version('check_zpools')}")
print(f"imported from {check_zpools.__file__}")

## 3) CLI: version and help

In [ ]:
!check_zpools --version
assert _exit_code == 0
!check_zpools --help
assert _exit_code == 0

## 4) CLI: metadata and the merged configuration

`config` shows what the layered merge actually produced - defaults, then the app,
host and user files, then `.env`, then the environment. It is the command to reach
for when a setting seems to be ignored.

In [ ]:
!check_zpools info
assert _exit_code == 0
!check_zpools config --format json --section zfs
assert _exit_code == 0

## 5) Driving it from a script: the JSON envelope

`check`, `scrub` and `service-status` all take `--format json`. Exit codes are the
contract: **0** healthy, **1** warnings, **2** critical.

In [ ]:
# No exit-code assert here: `check` reports this host's pool health (0/1/2) and fails
# without a working ZFS, so the result depends on the machine running the notebook.
!check_zpools check --format json

## 6) What a failure looks like

The cell above prints pool data on a host with ZFS. Anywhere else it prints an error
**object** rather than a traceback, naming the exception class so a caller can branch
on it instead of matching prose:

```json
{
  "error": {
    "type": "ZFSNotAvailableError",
    "message": "zpool command not found. Please install ZFS utilities."
  }
}
```

That keeps stdout parseable on the failure path, which is the difference between
"it ran and the answer is no" and "it could not run".

In [ ]:
# No exit-code assert here: `check` reports this host's pool health (0/1/2) and fails
# without a working ZFS, so the result depends on the machine running the notebook.
!check_zpools --traceback check

## 7) The CLI is also reachable as a module

In [ ]:
!{sys.executable} -m check_zpools info
assert _exit_code == 0

## 8) Python API: the validated configuration

`load_settings()` returns a validated `CheckZpoolsConfig`. Every section is a typed
model, so thresholds are attributes rather than dictionary keys - and a malformed
value in a config file falls back to the default and is logged rather than raising,
because a monitoring daemon must not die on a typo.

In [ ]:
from check_zpools.config import load_settings

settings = load_settings()

print(f"capacity warning:  {settings.zfs.capacity_warning_percent}%")
print(f"capacity critical: {settings.zfs.capacity_critical_percent}%")
print(f"scrub poll every:  {settings.scrub.poll_interval_seconds}s")
print(f"alert resend after: {settings.daemon.alert_resend_interval_hours}h")

## 9) Where to go next

- `check_zpools scrub` starts a scrub and waits for the pool rather than a clock -
  there is deliberately no timeout, since a large pool legitimately takes hours.
- `check_zpools service-install` runs the monitor as a systemd unit, and
  `service-status --format json` reports its state plus the alerts it is tracking.
- `check_zpools <command> --help` documents any subcommand.